In [ ]:
import pandas as pd
from shapely import wkt
import geopandas as gpd
from shapely import wkt
import subprocess, os
import numpy as np


os.environ["JAVA_HOME"] = subprocess.run(
    ["/usr/libexec/java_home", "-v", "21"],
    capture_output=True, text=True, check=True
).stdout.strip()


import r5py
import warnings
warnings.filterwarnings("ignore")

In [ ]:
bus_sol = pd.read_csv('Nodes/Bus-Sol.csv')
bus = pd.read_csv('Nodes/Bus.csv')
fgc_sol = pd.read_csv('Nodes/FGC-Sol.csv')
fgc = pd.read_csv('Nodes/FGC.csv')
metro_sol = pd.read_csv('Nodes/Metro-Sol.csv')
metro = pd.read_csv('Nodes/Metro.csv')
tram_sol = pd.read_csv('Nodes/Tram-Sol.csv')
tram = pd.read_csv('Nodes/Tram.csv')
destinations = pd.read_csv('Nodes/Destinations.csv')

all_stops = pd.concat([bus_sol, bus, fgc_sol, fgc, metro_sol, metro, tram_sol, tram], ignore_index=True)
all_stops['geometry']= all_stops['geometry'].apply(wkt.loads)
all_stops = gpd.GeoDataFrame(all_stops, geometry='geometry', crs="EPSG:4326")
#all_stops = all_stops[:100]


In [ ]:
metro_sol = pd.read_csv('Nodes/Metro-Sol.csv')
metro = pd.read_csv('Nodes/Metro.csv')

all_stops = pd.concat([metro_sol, metro], ignore_index=True)
all_stops['geometry']= all_stops['geometry'].apply(wkt.loads)
all_stops = gpd.GeoDataFrame(all_stops, geometry='geometry', crs="EPSG:4326")
#all_stops = all_stops[:100]


In [ ]:
pbf_path = os.path.abspath("Data/Neigh/cataluna-260811.osm.pbf")
transport_network = r5py.TransportNetwork(pbf_path)

In [ ]:
all_stops

In [31]:
import r5py
import r5py.sampledata.helsinki
import shapely

transport_network = r5py.TransportNetwork(
    r5py.sampledata.helsinki.osm_pbf,
    [
        r5py.sampledata.helsinki.gtfs,
    ]
)

RAILWAY_STATION = shapely.Point(24.941521, 60.170666)

import datetime

isochrones = r5py.Isochrones(
    transport_network,
    origins=RAILWAY_STATION,
    departure=datetime.datetime(2022, 2, 22, 8, 30),
    transport_modes=[r5py.TransportMode.TRANSIT, r5py.TransportMode.WALK],
    isochrones=[5, 10, 15],
)



In [32]:
isochrones['category'] = [5, 10, 15]
isochrones.drop(columns=['travel_time'], inplace=True)

In [33]:
isocrones_pol = isochrones.copy()
isocrones_pol['geometry'] = isocrones_pol.polygonize(node = True)
isocrones_pol

,geometry,category
0,"POLYGON ((24.93896 60.16979, 24.93873 60.16996...",5
1,"POLYGON ((24.92815 60.16807, 24.92793 60.16977...",10
2,"POLYGON ((24.91462 60.16416, 24.91514 60.16429...",15


In [36]:
isocrones_pol

,geometry,category
0,"POLYGON ((24.93896 60.16979, 24.93873 60.16996...",5
1,"POLYGON ((24.92815 60.16807, 24.92793 60.16977...",10
2,"POLYGON ((24.91462 60.16416, 24.91514 60.16429...",15


In [37]:
isochrones

,geometry,category
0,"MULTILINESTRING ((24.93896 60.16979, 24.93873 ...",5
1,"MULTILINESTRING ((24.92815 60.16807, 24.92793 ...",10
2,"MULTILINESTRING ((24.97562 60.18749, 24.97567 ...",15


In [34]:
import altair as alt

alt.Chart(isocrones_pol).mark_geoshape(filled=False).encode(
    color=alt.Color('category:N'))

alt.Chart(...)

In [35]:
import altair as alt

alt.Chart(isochrones).mark_geoshape(filled=False).encode(
    color=alt.Color('category:N'))

alt.Chart(...)

In [ ]:
buffers = all_stops[['id', 'geometry']].copy()
buffers.to_crs('EPSG:25831',inplace=True)
buffers['geometry'] = buffers.geometry.buffer(300)
all_stops.to_crs('EPSG:25831',inplace=True)

matches = gpd.sjoin(
    buffers.rename(columns={'id': 'origin_id'}),
    all_stops[['id', 'geometry']].rename(
        columns={'id': 'destination_id'}
    ),
    how='inner',
    predicate='intersects'
)

# Remove the stop matching with itself
matches = matches[
    matches['origin_id'] != matches['destination_id']
]

# Keep only the desired columns
close_stops = matches[
    ['origin_id', 'destination_id']
].reset_index(drop=True)

close_stops = pd.merge(close_stops, all_stops[['id', 'stop_id','stop_type','name','linia']], left_on='origin_id', right_on='id', how='left').drop(columns=['id'])
close_stops = close_stops[~close_stops['destination_id'].str.startswith('S')]
close_stops.rename(columns={'stop_id': 'origin_stop_id', 'stop_type': 'origin_stop_type','name': 'origin_stop_name', 'linia': 'origin_linia'}, inplace=True)
close_stops = pd.merge(close_stops, all_stops[['id', 'stop_id','stop_type','name','linia']], left_on='destination_id', right_on='id', how='left').drop(columns=['id'])
close_stops.rename(columns={'stop_id': 'destination_stop_id', 'stop_type': 'destination_stop_type','name': 'destination_stop_name', 'linia': 'destination_linia'}, inplace=True)
close_stops = close_stops[['origin_id', 'origin_stop_id', 'origin_stop_type', 'origin_stop_name', 'origin_linia', 'destination_id', 'destination_stop_id', 'destination_stop_type', 'destination_stop_name', 'destination_linia']]
close_stops['tram'] = close_stops['origin_stop_name'] + ' - ' + close_stops['destination_stop_name']
close_stops['mode'] = close_stops['origin_stop_type'] + ' - ' + close_stops['destination_stop_type']
close_stops['lines'] = close_stops['origin_linia'] + ' - ' + close_stops['destination_linia']
close_stops['type'] = 'Exchange'
close_stops.drop(columns=['origin_stop_name', 'destination_stop_name','origin_stop_type','destination_stop_type','origin_linia','destination_linia'], inplace=True)
all_stops.to_crs('EPSG:4326',inplace=True)

In [ ]:
close_stops

In [ ]:
same_stops = close_stops[
    close_stops['origin_stop_id'] == close_stops['destination_stop_id']
].copy()
same_stops = same_stops.drop(columns=['origin_stop_id', 'destination_stop_id'])

conditions = [
    same_stops['origin_id'].str.startswith('S', na=False) & same_stops['destination_id'].str.startswith(('M', 'F'), na=False),
    same_stops['origin_id'].str.startswith('S', na=False) & same_stops['destination_id'].str.startswith('B', na=False),
    same_stops['origin_id'].str.startswith('S', na=False) & same_stops['destination_id'].str.startswith('T', na=False),
    same_stops['origin_id'].str.startswith('B', na=False) & same_stops['destination_id'].str.startswith('B', na=False),
    same_stops['origin_id'].str.startswith('M', na=False) & same_stops['destination_id'].str.startswith('M', na=False),
    same_stops['origin_id'].str.startswith('T', na=False) & same_stops['destination_id'].str.startswith('T', na=False),
]
choices = [3, 1, 2, 1, 2, 1]

same_stops['exchange_time'] = np.select(conditions, choices, default=0)
same_stops

#close_stops.drop(columns=['origin_stop_name', 'destination_stop_name','origin_stop_type','destination_stop_type','origin_linia','destination_linia'], inplace=True)

In [ ]:
import os
pbf_path = os.path.abspath("Data/Neigh/cataluna-260811.osm.pbf")
transport_network = r5py.TransportNetwork(pbf_path)

In [ ]:
different_stops = close_stops[
    (close_stops['origin_stop_id'] != close_stops['destination_stop_id'])
]
different_stops.drop(columns=['origin_stop_id', 'destination_stop_id'], inplace=True)

itineraries = pd.DataFrame()
process = 0
for _, row in different_stops[:200].iterrows():
    process += 1
    print(f"Processing {process}/{len(different_stops)}")
    origin_id = row['origin_id']
    destination_id = row['destination_id']
    origins = all_stops[all_stops['id'] == origin_id].copy()
    destinations = all_stops[all_stops['id'] == destination_id].copy()
    origins.rename(columns={'origin_id': 'id'}, inplace=True)
    destinations.rename(columns={'destination_id': 'id'}, inplace=True)
    origins = gpd.GeoDataFrame(origins, geometry='geometry', crs="EPSG:4326")
    destinations = gpd.GeoDataFrame(destinations, geometry='geometry', crs="EPSG:4326")
    detailed_itineraries = r5py.DetailedItineraries(
        transport_network,
        origins=origins,
        destinations=destinations,
        transport_modes=[r5py.TransportMode.WALK],
        snap_to_network=False,
    )
    itineraries = pd.concat([itineraries, detailed_itineraries], ignore_index=True)
    


In [ ]:
all_stops[all_stops['id'] =='F-L6-PR']

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import folium.plugins
import shapely.wkt as wkt


# ---------------------------------------------------------
# Load neighbourhoods
# ---------------------------------------------------------

neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)

neigh = neigh[["name", "geometry"]]

neigh["geometry"] = neigh.geometry.buffer(0)


# ---------------------------------------------------------
# Prepare itinerary data
# ---------------------------------------------------------

itineraries["mode"] = itineraries["transport_mode"].astype(str)

itineraries["travel time (min)"] = itineraries["travel_time"].apply(
    lambda t: round(t.total_seconds() / 60.0, 2)
)

itineraries["trip"] = itineraries.apply(
    lambda row: f"{row['from_id']} → {row['to_id']}",
    axis=1
)


# ---------------------------------------------------------
# Create map
# ---------------------------------------------------------

detailed_routes_map = (
    itineraries[
        [
            "geometry",
            "distance",
            "mode",
            "travel time (min)",
            "from_id",
            "to_id",
            "trip",
            "option",
            "segment",
        ]
    ]
    .explore(
        tooltip=[
            "trip",
            "option",
            "segment",
            "mode",
            "travel time (min)",
            "distance",
        ],
        column="mode",
        tiles="CartoDB.Positron",
        style_kwds={
            "weight": 3,
            "opacity": 0.8,
        },
        highlight_kwds={
            "weight": 6,
            "opacity": 1,
        },
    )
)


# ---------------------------------------------------------
# Get stops involved in the itineraries
# ---------------------------------------------------------

stop_ids = pd.unique(
    pd.concat([
        itineraries["from_id"],
        itineraries["to_id"]
    ])
)

stops_map = all_stops[
    all_stops["id"].isin(stop_ids)
].copy()

stops_map = gpd.GeoDataFrame(
    stops_map,
    geometry="geometry",
    crs="EPSG:4326"
)


# ---------------------------------------------------------
# Assign neighbourhood to each stop
# ---------------------------------------------------------

# Project both layers for the spatial join
stops_projected = stops_map.to_crs("EPSG:25831")
neigh_projected = neigh.to_crs("EPSG:25831")

stops_map = gpd.sjoin(
    stops_projected,
    neigh_projected[["name", "geometry"]],
    how="left",
    predicate="within"
)

# Return to WGS84 for Folium
stops_map = stops_map.to_crs("EPSG:4326")


# ---------------------------------------------------------
# Add stop markers
# ---------------------------------------------------------

stops_map.apply(
    lambda row: folium.Marker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],
        tooltip=(
            f"Stop: {row['id']}<br>"
        ),
        icon=folium.plugins.BeautifyIcon(
            icon_shape="marker",
            number=str(row["id"]),
            border_color="#728224",
            text_color="#728224",
        ),
    ).add_to(detailed_routes_map),
    axis=1,
)

neigh.explore(
    m=detailed_routes_map,
    color="grey",
    fill=False,
    style_kwds={
        "weight": 1,
        "opacity": 0.5,
    },
    tooltip=["name"],
)


detailed_routes_map

In [ ]:
stops_map

In [ ]:
different_stops['origin_stop_id'].unique()